In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import json
import re
import ast

class QWen2Evaluator:
    # 加载模型及token
    def __init__(self, model_path):
        self.model_path = model_path
        self.model = AutoModelForCausalLM.from_pretrained(self.model_path, device_map="auto", torch_dtype="auto")
        self.model = self.model.eval()
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
    # 调用模型根据Promt（query）输出答案
    def answer(self, query, temperature=None):
        text = self.tokenizer.apply_chat_template(
            query,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to("cuda")
        if temperature:
            generated_ids = self.model.generate(
                model_inputs.input_ids,
                max_new_tokens=1024,
                temperature=temperature
            )
        else:
            generated_ids = self.model.generate(
                model_inputs.input_ids,
                max_new_tokens=1024
            )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return response

def get_answer(model_path):
    evaluator = QWen2Evaluator(model_path)
    prompt = '你是一名文学专家，请根据用户的问题推荐合适的图书。问题：' + '想通过书籍理解当代年轻人的低欲望生活现象，推荐哪些社会纪实作品？'
    keyword_rec = evaluator.answer([{"role": "user", "content": '<s>Human:' + prompt + '</s>'}])
    print(keyword_rec)

if __name__ == "__main__":
    device = "cuda" 
    model_path = "/root/autodl-tmp/"
    model_name = 'llm_rc'
    get_answer(model_path + model_name)
    
    print('**************************************** 模型测试结束！****************************************')

/root/miniconda3/envs/llama/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.02it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


<s>Assistant:《无业青年》《零世代》《不婚主义》《低欲望社会》《懒人经济》</s>
**************************************** 模型测试结束！****************************************
